In [1]:
import os
import sys

import pandas as pd
import numpy as np
import anndata as ad
import tacco as tc

# The notebook expects to be executed either in the workflow directory or in the repository root folder...
sys.path.insert(1, os.path.abspath('workflow' if os.path.exists('workflow/common_code.py') else '..')) 
import common_code

In [2]:
import numpy as np
import scipy.sparse as sp

def make_float32(adata):
    # If it's a view, materialize it
    if adata.is_view:
        adata = adata.copy()

    # If the AnnData was opened in backed mode, reload into memory
    if getattr(adata, "isbacked", False):
        import anndata as ad
        adata = ad.read_h5ad(adata.filename, backed=None)

    # Convert X
    if sp.issparse(adata.X):
        adata.X = adata.X.asfptype()           # to float
        adata.X = adata.X.astype(np.float32)   # to float32
    else:
        adata.X = np.asarray(adata.X, dtype=np.float32)

    return adata

In [3]:
import os.path as osp 
import scanpy as sc
rna_path = '/maiziezhou_lab2/yuling/MERFISH_spinal_cord_resolved_0718.h5ad'
rna = sc.read_h5ad(rna_path)
rows = []
base = '/maiziezhou_lab2/yuling/MouseSpinal/label_transfer/tacco/scNucleus_output'
unique_section = rna.obs['Section ID'].unique()
selected_0503 = [s for s in unique_section if s.startswith('0503')]
selected_0503_clean = [s for s in selected_0503 if s != "0503_nan_nan"]
selected_0503_1 = [s for s in selected_0503_clean if s != "0503_F4_C"]
# Output base path
 
os.makedirs(base, exist_ok=True)
puck = rna[rna.obs['Section ID'] == '0503_F4_C'].copy()

In [4]:
reference = sc.read_h5ad('/maiziezhou_lab2/yuling/datasets/obj_integrated_sc_nucleus.h5ad')

In [5]:
import os
import sys
import time
import resource

import numpy as np
import scipy.sparse as sp
import pandas as pd

base = "/maiziezhou_lab2/yuling/MouseSpinal/label_transfer/tacco/scNucleus_output"
os.makedirs(base, exist_ok=True)

def make_float32(adata):
    # If it's a view, materialize it
    if adata.is_view:
        adata = adata.copy()

    # If the AnnData was opened in backed mode, reload into memory
    if getattr(adata, "isbacked", False):
        import anndata as ad
        adata = ad.read_h5ad(adata.filename, backed=None)

    # Convert X
    if sp.issparse(adata.X):
        adata.X = adata.X.asfptype()           # to float
        adata.X = adata.X.astype(np.float32)   # to float32
    else:
        adata.X = np.asarray(adata.X, dtype=np.float32)

    return adata

t0 = time.perf_counter()

# --- Apply to both datasets ---
puck = make_float32(puck)
adata_subset = make_float32(reference)

# (Optional) If your counts actually live in a layer, move it into X *as float32*
if 'counts' in puck.layers:
    L = puck.layers['counts']
    puck.X = (L.asfptype().astype(np.float32) if sp.issparse(L)
              else np.asarray(L, dtype=np.float32))

if 'counts' in adata_subset.layers:
    L = adata_subset.layers['counts']
    adata_subset.X = (L.asfptype().astype(np.float32) if sp.issparse(L)
                      else np.asarray(L, dtype=np.float32))

# Sanity checks
print("puck.X dtype:", puck.X.dtype)
print("adata_subset.X dtype:", adata_subset.X.dtype)

# Force TACCO to read from X (not a layer)
import tacco as tc
tc.tl.annotate(
    puck,
    adata_subset,
    'final_cluster_assignment',
    result_key='ClusterName',
    counts_location='X',   # be explicit
)
tmp = puck.obsm['ClusterName']
max_col = tmp.idxmax(axis=1).rename('max_col')
max_val = tmp.max(axis=1).rename('max_val')
out = pd.concat([max_col, max_val], axis=1) 
out.index = out.index.astype(puck.obs.index.dtype)
merged = puck.obs.join(out, how='left')          
# merged = puck.obs.merge(out, left_index=True, right_index=True, how='left')

print(merged[["max_col", "max_val"]].head())
outfile = os.path.join(base, "TACCO_pred.csv")
merged.to_csv(outfile)

elapsed_sec = time.perf_counter() - t0
rss_peak = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
if sys.platform == "darwin":
    peak_mib = rss_peak / (1024.0 ** 2)
else:
    peak_mib = rss_peak / 1024.0

metrics_path = os.path.join(base, "runtimeSec_memoryMiB.csv")
pd.DataFrame([{"Time": elapsed_sec, "Memory": peak_mib}]).to_csv(metrics_path, index=False)
print("Saved:", outfile)
print("Saved metrics:", metrics_path)

puck.X dtype: float32
adata_subset.X dtype: float32
Starting preprocessing
Annotation profiles were not found in `reference.varm["final_cluster_assignment"]`. Constructing reference profiles with `tacco.preprocessing.construct_reference_profiles` and default arguments...
Finished preprocessing in 0.73 seconds.
Starting annotation of data with shape (6602, 428) and a reference of shape (52623, 428) using the following wrapped method:
+- platform normalization: platform_iterations=0, gene_keys=final_cluster_assignment, normalize_to=adata
   +- multi center: multi_center=None multi_center_amplitudes=True
      +- bisection boost: bisections=4, bisection_divisor=3
         +- core: method=OT annotation_prior=None
mean,std( rescaling(gene) )  4.771964152390818 16.621973849614427
bisection run on 1
bisection run on 0.6666666666666667
bisection run on 0.4444444444444444
bisection run on 0.2962962962962963
bisection run on 0.19753086419753085
bisection run on 0.09876543209876543
Finished annot

In [ ]:
import pandas as pd
tmp = puck.obsm['ClusterName']
max_col = tmp.idxmax(axis=1).rename('max_col')
max_val = tmp.max(axis=1).rename('max_val')
out = pd.concat([max_col, max_val], axis=1) 
print(out.head())

                          max_col   max_val
2787445900215100190  Astrocytes-2  1.000000
2787445900215100193  Astrocytes-2  1.000000
2787445900215100194  Astrocytes-2  0.932434
2787445900215100198  Astrocytes-2  0.403368
2787445900216100058  Astrocytes-2  0.878864


In [8]:
out.index = out.index.astype(puck.obs.index.dtype)

In [ ]:
merged = puck.obs.join(out, how='left')          
# merged = puck.obs.merge(out, left_index=True, right_index=True, how='left')

print(merged[['max_col','max_val']].head())

                          max_col   max_val
2787445900215100190  Astrocytes-2  1.000000
2787445900215100193  Astrocytes-2  1.000000
2787445900215100194  Astrocytes-2  0.932434
2787445900215100198  Astrocytes-2  0.403368
2787445900216100058  Astrocytes-2  0.878864


In [11]:
outfile = os.path.join('/maiziezhou_lab2/yuling/MouseSpinal/label_transfer/tacco/scNucleus_output', f"TACCO_pred.csv")
merged.to_csv(outfile)